# Aplicação 2 — Rede Neural Convolucional (CNN) com o CIFAR-10

**Objetivo:** mostrar na prática por que as redes convolucionais funcionam tão bem com imagens.

Vamos treinar **duas redes nos mesmos dados**:

1. uma **MLP** (rede totalmente conectada, igual à da primeira parte da aula), que "achata" a imagem;
2. uma **CNN**, que usa convolução, pesos compartilhados e pooling.

No final, comparamos as duas: acurácia, número de parâmetros e tempo de treino. Depois abrimos a CNN para ver **o que ela enxerga** em cada camada.

**O dataset — CIFAR-10:** 60 mil fotos coloridas de 32×32 pixels, divididas em 10 classes (avião, automóvel, pássaro, gato, cervo, cachorro, sapo, cavalo, navio e caminhão).

> ⚙️ **Antes de começar:** ative a GPU no Colab em **Ambiente de execução → Alterar o tipo de ambiente de execução → GPU (T4)**. Com GPU, o notebook inteiro roda em poucos minutos. Sem GPU, mude `MODO_RAPIDO = True` na célula de configurações.

## 0. Configurações

Todos os hiperparâmetros da demonstração ficam aqui, para facilitar os testes durante a aula.

- **Época:** uma passada completa por todas as imagens de treino.
- **Tamanho do lote (batch size):** quantas imagens a rede vê antes de cada atualização dos pesos.
- **MODO_RAPIDO:** usa só 10 mil imagens de treino e menos épocas, para rodar mesmo sem GPU.

In [ ]:
MODO_RAPIDO = False      # True = versão mais leve (use se não houver GPU)

EPOCAS_MLP = 10          # épocas de treino da MLP
EPOCAS_CNN = 20          # máximo de épocas da CNN (o early stopping pode parar antes)
TAMANHO_LOTE = 64        # imagens por atualização dos pesos
SEMENTE = 42             # deixa os resultados reproduzíveis

if MODO_RAPIDO:
    EPOCAS_MLP, EPOCAS_CNN = 5, 8

## 1. Importando as bibliotecas

- **TensorFlow/Keras:** monta e treina as redes neurais.
- **NumPy:** operações com matrizes (as imagens são matrizes de números).
- **Matplotlib:** gráficos e imagens.
- **scikit-learn:** matriz de confusão.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

keras.utils.set_random_seed(SEMENTE)

print("TensorFlow:", tf.__version__)
if tf.config.list_physical_devices("GPU"):
    print("GPU disponível ✔")
else:
    print("Sem GPU: ative em 'Ambiente de execução > Alterar o tipo de ambiente de execução'"
          " ou use MODO_RAPIDO = True.")

## 2. Carregando o CIFAR-10

O Keras já traz o CIFAR-10 separado em **treino** (50 mil imagens) e **teste** (10 mil imagens).
Cada imagem tem formato **32 × 32 × 3**: largura, altura e os 3 canais de cor (R, G, B).

In [ ]:
(x_treino_total, y_treino_total), (x_teste, y_teste) = keras.datasets.cifar10.load_data()

# os rótulos vêm como uma coluna (n, 1); transformamos em um vetor simples (n,)
y_treino_total = y_treino_total.flatten()
y_teste = y_teste.flatten()

CLASSES = ["avião", "automóvel", "pássaro", "gato", "cervo",
           "cachorro", "sapo", "cavalo", "navio", "caminhão"]

print("Treino:", x_treino_total.shape, "| Teste:", x_teste.shape)
print("Valores dos pixels: de", x_treino_total.min(), "a", x_treino_total.max())

## 3. Olhando os dados

Primeiro, alguns exemplos de cada classe. Repare que as imagens são pequenas (32×32) e bem variadas: ângulo, cor, fundo e tamanho do objeto mudam bastante.

**Ligação com os slides:** *"CNN: a ideia"* — para a rede, uma imagem colorida é um **volume** de números com 3 camadas (os canais R, G e B).

In [ ]:
fig, eixos = plt.subplots(3, 10, figsize=(15, 5))
for classe in range(10):
    indices = np.where(y_treino_total == classe)[0][:3]
    for linha, i in enumerate(indices):
        eixos[linha, classe].imshow(x_treino_total[i])
        eixos[linha, classe].axis("off")
    eixos[0, classe].set_title(CLASSES[classe], fontsize=11)
plt.suptitle("Exemplos do CIFAR-10", fontsize=14)
plt.tight_layout()
plt.show()

Agora, uma única imagem separada nos seus **3 canais de cor**. A rede não "vê" a foto: ela recebe estas três matrizes de 32 × 32 números, empilhadas.

In [ ]:
img = x_treino_total[np.where(y_treino_total == CLASSES.index("navio"))[0][0]]

fig, eixos = plt.subplots(1, 4, figsize=(13, 3.5))
eixos[0].imshow(img)
eixos[0].set_title("Imagem original\n(32 × 32 × 3)")
for c, (nome, mapa) in enumerate([("R (vermelho)", "Reds"), ("G (verde)", "Greens"), ("B (azul)", "Blues")]):
    eixos[c + 1].imshow(img[:, :, c], cmap=mapa)
    eixos[c + 1].set_title(f"Canal {nome}\n(32 × 32)")
for e in eixos:
    e.axis("off")
plt.tight_layout()
plt.show()

print("Canto superior esquerdo do canal R (5 × 5 pixels):")
print(img[:5, :5, 0])

## 4. Preparando os dados

Três passos:

1. **Normalizar os pixels** de 0–255 para 0–1. Valores pequenos e na mesma escala ajudam a descida do gradiente.
2. **Separar uma validação:** as últimas 5 mil imagens do treino viram o conjunto de **validação**, usado para acompanhar o treino e para o early stopping. O **teste** fica guardado para a avaliação final.
3. **One-hot:** cada rótulo vira um vetor de 10 posições, com 1 na classe certa (ex.: gato = classe 3 → `[0,0,0,1,0,0,0,0,0,0]`).

**Ligação com os slides:** *"Treino, validação e teste"* (lista de exercícios, simulado e prova) e *"MNIST: dados, entrada e saída"* (one-hot).

In [ ]:
# 1) normalização
x_treino_total = x_treino_total.astype("float32") / 255.0
x_teste = x_teste.astype("float32") / 255.0

# 2) treino / validação / teste
x_treino, y_treino = x_treino_total[:-5000], y_treino_total[:-5000]
x_val, y_val = x_treino_total[-5000:], y_treino_total[-5000:]
if MODO_RAPIDO:
    x_treino, y_treino = x_treino[:10000], y_treino[:10000]

# 3) one-hot
Y_treino = keras.utils.to_categorical(y_treino, 10)
Y_val = keras.utils.to_categorical(y_val, 10)
Y_teste = keras.utils.to_categorical(y_teste, 10)

print(f"Treino: {len(x_treino)} | Validação: {len(x_val)} | Teste: {len(x_teste)}")
print(f"Rótulo {y_treino[0]} ({CLASSES[y_treino[0]]}) em one-hot: {Y_treino[0]}")
print(f"Passos (atualizações dos pesos) por época: {int(np.ceil(len(x_treino) / TAMANHO_LOTE))}")

## 5. Referência: uma MLP (rede totalmente conectada)

Antes da CNN, treinamos a rede da primeira parte da aula:

- `Flatten` **achata** a imagem 32 × 32 × 3 em uma fila de **3.072 números**: a informação de quem é vizinho de quem se perde;
- duas camadas ocultas densas com **ReLU**;
- saída com **softmax**: 10 probabilidades, uma por classe.

Função de custo: **entropia cruzada categórica**. Otimizador: **Adam** (uma variação da descida do gradiente que ajusta o tamanho do passo sozinha).

**Ligação com os slides:** *"Por que a MLP sofre com imagens?"* — veja no resumo quantos parâmetros a primeira camada tem.

In [ ]:
mlp = keras.Sequential([
    keras.Input(shape=(32, 32, 3)),
    layers.Flatten(),                          # 32 x 32 x 3 -> 3.072 entradas
    layers.Dense(256, activation="relu"),      # camada oculta 1
    layers.Dense(128, activation="relu"),      # camada oculta 2
    layers.Dense(10, activation="softmax"),    # 10 classes
], name="MLP")

mlp.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
mlp.summary()

In [ ]:
inicio = time.time()
hist_mlp = mlp.fit(x_treino, Y_treino,
                   validation_data=(x_val, Y_val),
                   epochs=EPOCAS_MLP, batch_size=TAMANHO_LOTE, verbose=2)
tempo_mlp = time.time() - inicio
print(f"\nTempo de treino da MLP: {tempo_mlp:.0f} s")

## 6. A CNN

A arquitetura segue o padrão visto nos slides: **[Convolução → ReLU → Pooling] × 3 → Flatten → Densa → Softmax**.

| Camada | O que faz | Saída |
|---|---|---|
| `Conv2D(32, 3×3)` | 32 filtros 3×3 percorrem a imagem (campos receptivos locais + pesos compartilhados) | 32 × 32 × 32 |
| `MaxPooling2D(2×2)` | guarda o maior valor de cada região 2×2 | 16 × 16 × 32 |
| `Conv2D(64, 3×3)` + pooling | 64 filtros sobre os mapas anteriores | 8 × 8 × 64 |
| `Conv2D(128, 3×3)` + pooling | 128 filtros | 4 × 4 × 128 |
| `Flatten` | agora sim achata, **depois** de extrair as características espaciais | 2.048 |
| `Dropout(0.5)` | desliga metade dos neurônios ao acaso, **só no treino** | 2.048 |
| `Dense(128)` + `Dense(10, softmax)` | classificador | 10 |

Repare que, a cada bloco, os mapas ficam **menores e mais numerosos**: a rede troca "onde" por "o quê" (bordas → formas → partes de objetos).

`padding="same"` acrescenta bordas de zeros para a saída da convolução ter o mesmo tamanho da entrada.

**Ligação com os slides:** *"Campos receptivos locais"*, *"Pesos compartilhados"*, *"Pooling"*, *"Juntando tudo"* e *"Dropout"*.

In [ ]:
cnn = keras.Sequential([
    keras.Input(shape=(32, 32, 3)),

    layers.Conv2D(32, (3, 3), activation="relu", padding="same", name="conv1"),
    layers.MaxPooling2D((2, 2), name="pool1"),

    layers.Conv2D(64, (3, 3), activation="relu", padding="same", name="conv2"),
    layers.MaxPooling2D((2, 2), name="pool2"),

    layers.Conv2D(128, (3, 3), activation="relu", padding="same", name="conv3"),
    layers.MaxPooling2D((2, 2), name="pool3"),

    layers.Flatten(name="flatten"),
    layers.Dropout(0.5, name="dropout"),
    layers.Dense(128, activation="relu", name="densa"),
    layers.Dense(10, activation="softmax", name="saida"),
], name="CNN")

cnn.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
cnn.summary()

print(f"\nParâmetros — MLP: {mlp.count_params():,} | CNN: {cnn.count_params():,}".replace(",", "."))
print("Exemplo: conv1 tem 32 filtros × (3×3×3 pesos + 1 bias) =", 32 * (3 * 3 * 3 + 1), "parâmetros")

## 7. Treinando a CNN (com early stopping)

O treino é igual ao da MLP: forward pass, cálculo do erro, **backpropagation** e atualização dos pesos, lote a lote.

Acrescentamos o **early stopping**: se a perda na validação não melhorar por 3 épocas seguidas (a "paciência"), o treino para e a rede volta para os pesos da **melhor** época.

**Ligação com os slides:** *"Treino, validação e teste + early stopping"*.

> ⏱️ Com GPU, leva por volta de 1 a 2 minutos. Uma dica para a aula: deixe esta célula rodando enquanto explica a arquitetura.

In [ ]:
parada_antecipada = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=3, restore_best_weights=True, verbose=1)

inicio = time.time()
hist_cnn = cnn.fit(x_treino, Y_treino,
                   validation_data=(x_val, Y_val),
                   epochs=EPOCAS_CNN, batch_size=TAMANHO_LOTE,
                   callbacks=[parada_antecipada], verbose=2)
tempo_cnn = time.time() - inicio
print(f"\nTempo de treino da CNN: {tempo_cnn:.0f} s")

## 8. Curvas de aprendizado: treino × validação

Como ler os gráficos:

- se as curvas de **treino** e **validação** andam juntas, a rede está **generalizando**;
- se a de treino continua melhorando e a de validação para ou piora, começou o **overfitting** (a rede está decorando).

**Ligação com os slides:** *"Overfitting: decorar não é aprender"*.

In [ ]:
def plotar_historico(hist, titulo):
    h = hist.history
    epocas = range(1, len(h["loss"]) + 1)
    fig, (e1, e2) = plt.subplots(1, 2, figsize=(12, 4))
    e1.plot(epocas, h["accuracy"], "o-", label="treino")
    e1.plot(epocas, h["val_accuracy"], "o-", label="validação")
    e1.set_title(f"{titulo} — acurácia"); e1.set_xlabel("época"); e1.legend(); e1.grid(alpha=.3)
    e2.plot(epocas, h["loss"], "o-", label="treino")
    e2.plot(epocas, h["val_loss"], "o-", label="validação")
    e2.set_title(f"{titulo} — perda (entropia cruzada)"); e2.set_xlabel("época"); e2.legend(); e2.grid(alpha=.3)
    plt.tight_layout()
    plt.show()

plotar_historico(hist_mlp, "MLP")
plotar_historico(hist_cnn, "CNN")

## 9. MLP × CNN no conjunto de teste

Agora, a "prova": as 10 mil imagens de teste, que nenhuma das redes viu durante o treino.

In [ ]:
_, acc_mlp = mlp.evaluate(x_teste, Y_teste, verbose=0)
_, acc_cnn = cnn.evaluate(x_teste, Y_teste, verbose=0)

def fmt(n):
    return f"{n:,}".replace(",", ".")

print(f"{'Modelo':<8}{'Parâmetros':>14}{'Acurácia no teste':>20}{'Tempo de treino':>18}")
def pct(v):
    return f"{v * 100:.1f}%".replace(".", ",")

print(f"{'MLP':<8}{fmt(mlp.count_params()):>14}{pct(acc_mlp):>20}{tempo_mlp:>16.0f} s")
print(f"{'CNN':<8}{fmt(cnn.count_params()):>14}{pct(acc_cnn):>20}{tempo_cnn:>16.0f} s")
print(f"\nAcaso (chutar entre 10 classes): 10,0%")

fig, (e1, e2) = plt.subplots(1, 2, figsize=(10, 3.8))
cores = ["#9aa5b1", "#2b6cb0"]
e1.bar(["MLP", "CNN"], [acc_mlp * 100, acc_cnn * 100], color=cores)
e1.axhline(10, ls="--", c="gray", lw=1); e1.text(1.45, 11, "acaso", color="gray", ha="right")
e1.set_ylim(0, 100); e1.set_title("Acurácia no teste (%)")
for i, v in enumerate([acc_mlp * 100, acc_cnn * 100]):
    e1.text(i, v + 2, pct(v / 100), ha="center", fontweight="bold")
params = [mlp.count_params() / 1000, cnn.count_params() / 1000]
e2.bar(["MLP", "CNN"], params, color=cores)
e2.set_title("Parâmetros (milhares)"); e2.set_ylim(0, max(params) * 1.15)
for i, v in enumerate(params):
    e2.text(i, v * 1.02, f"{v:.0f} mil", ha="center", fontweight="bold")
plt.tight_layout()
plt.show()

**O que esperar:** a CNN costuma acertar bem mais que a MLP (na faixa de 70–78% contra 45–52% no CIFAR-10), **mesmo com menos parâmetros**. É a demonstração prática de que aproveitar a estrutura espacial da imagem vale mais do que simplesmente ter mais pesos.

## 10. Onde a CNN erra?

**Matriz de confusão:** cada linha é a classe real e cada coluna, a classe prevista. A diagonal são os acertos; fora dela, as confusões. Normalmente aparecem confusões "compreensíveis", como gato ↔ cachorro e automóvel ↔ caminhão.

In [ ]:
prob_cnn = cnn.predict(x_teste, verbose=0)
pred_cnn = prob_cnn.argmax(axis=1)

matriz = confusion_matrix(y_teste, pred_cnn)
fig, eixo = plt.subplots(figsize=(8, 8))
ConfusionMatrixDisplay(matriz, display_labels=CLASSES).plot(
    ax=eixo, cmap="Blues", colorbar=False, xticks_rotation=45)
eixo.set_title("Matriz de confusão da CNN (teste)")
eixo.set_xlabel("classe prevista"); eixo.set_ylabel("classe real")
plt.tight_layout()
plt.show()

Alguns erros, com a confiança da rede (a probabilidade que a softmax deu à classe escolhida):

In [ ]:
erros = np.where(pred_cnn != y_teste)[0][:10]
fig, eixos = plt.subplots(2, 5, figsize=(14, 6))
for e, i in zip(eixos.flat, erros):
    e.imshow(x_teste[i]); e.axis("off")
    e.set_title(f"real: {CLASSES[y_teste[i]]}\nprevisto: {CLASSES[pred_cnn[i]]} ({prob_cnn[i].max():.0%})",
                fontsize=10, color="crimson")
plt.suptitle("Exemplos de erros da CNN", fontsize=14)
plt.tight_layout()
plt.show()

E as **10 probabilidades da softmax** para uma imagem: a classe escolhida é a de maior probabilidade, e todas somam 1.

**Ligação com os slides:** *"Softmax + qual função usar"*.

In [ ]:
i = 0
fig, (e1, e2) = plt.subplots(1, 2, figsize=(11, 3.5), gridspec_kw={"width_ratios": [1, 2.5]})
e1.imshow(x_teste[i]); e1.axis("off")
e1.set_title(f"real: {CLASSES[y_teste[i]]}")
cores = ["#2b6cb0" if c == pred_cnn[i] else "#cbd5e0" for c in range(10)]
e2.bar(CLASSES, prob_cnn[i], color=cores)
soma = f"{prob_cnn[i].sum():.2f}".replace(".", ",")
e2.set_ylim(0, 1); e2.set_title(f"Saída da softmax (soma = {soma})")
e2.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

## 11. O que a CNN "vê"?

Abrimos a rede e mostramos os **mapas de características**: o resultado de aplicar os filtros de cada camada convolucional numa imagem.

- **conv1:** mapas parecidos com a imagem, destacando bordas, cores e contrastes simples;
- **conv2:** padrões mais combinados (texturas, cantos);
- **conv3:** mapas pequenos (8 × 8) e abstratos, que indicam **se** certas partes de objetos estão presentes, e não mais **onde** exatamente.

**Ligação com os slides:** *"Por que profundidade?"* (bordas → formas → objetos) e *"Pesos compartilhados"* (cada mapa = um filtro aplicado na imagem inteira).

In [ ]:
# escolhe uma imagem de teste que a CNN acertou (de preferência um cavalo)
acertos = np.where((pred_cnn == y_teste) & (y_teste == CLASSES.index("cavalo")))[0]
idx = int(acertos[0]) if len(acertos) else 0
imagem = x_teste[idx:idx + 1]

camadas = ["conv1", "conv2", "conv3"]
extrator = keras.Model(inputs=cnn.inputs,
                       outputs=[cnn.get_layer(nome).output for nome in camadas])
mapas = extrator.predict(imagem, verbose=0)

plt.figure(figsize=(3, 3))
plt.imshow(imagem[0]); plt.axis("off")
plt.title(f"Imagem de entrada: {CLASSES[y_teste[idx]]}")
plt.show()

n = 8  # mapas mostrados por camada (os 8 mais ativos, para evitar mapas "apagados" pela ReLU)
fig, eixos = plt.subplots(len(camadas), n, figsize=(16, 6.5))
for linha, (nome, mapa) in enumerate(zip(camadas, mapas)):
    mais_ativos = mapa[0].mean(axis=(0, 1)).argsort()[::-1][:n]
    for col, k in enumerate(mais_ativos):
        eixos[linha, col].imshow(mapa[0, :, :, k], cmap="viridis")
        eixos[linha, col].axis("off")
    eixos[linha, 0].set_title(f"{nome}: {mapa.shape[1]}×{mapa.shape[2]}, {mapa.shape[3]} mapas",
                              loc="left", fontsize=11)
plt.suptitle("Mapas de características (os 8 mais ativos de cada camada)", fontsize=14)
plt.tight_layout()
plt.show()

E os **32 filtros da primeira camada** (cada um com 3 × 3 pixels × 3 canais de cor). Eles foram **aprendidos** pelo treino, ninguém os desenhou. Mesmo pequenos, dá para notar filtros sensíveis a cores e a transições de claro para escuro.

In [ ]:
filtros, _ = cnn.get_layer("conv1").get_weights()       # formato: (3, 3, 3, 32)
filtros = (filtros - filtros.min()) / (filtros.max() - filtros.min())

fig, eixos = plt.subplots(2, 16, figsize=(16, 2.4))
for k, e in enumerate(eixos.flat):
    e.imshow(filtros[:, :, :, k], interpolation="nearest")
    e.axis("off")
plt.suptitle("Filtros aprendidos pela conv1", fontsize=13)
plt.tight_layout()
plt.show()

## 12. Teste com uma imagem sua (opcional)

Envie uma foto (de preferência com o objeto centralizado e de uma das 10 classes). Ela é reduzida para 32 × 32 pixels, o mesmo tamanho das imagens de treino, e a CNN mostra as 3 classes mais prováveis.

> Lembre que a rede só conhece as 10 classes do CIFAR-10: uma foto de algo diferente vai ser forçada a uma delas.

In [ ]:
try:
    from google.colab import files
    from PIL import Image

    enviados = files.upload()
    for nome in enviados:
        foto = Image.open(nome).convert("RGB")
        pequena = foto.resize((32, 32))
        entrada = np.asarray(pequena, dtype="float32")[None] / 255.0
        prob = cnn.predict(entrada, verbose=0)[0]
        top3 = prob.argsort()[::-1][:3]

        fig, (e1, e2) = plt.subplots(1, 2, figsize=(8, 3.5))
        e1.imshow(foto); e1.axis("off"); e1.set_title("Sua imagem")
        e2.imshow(pequena); e2.axis("off"); e2.set_title("Como a rede vê (32 × 32)")
        plt.show()
        for k in top3:
            print(f"{CLASSES[k]:>10}: {prob[k]:.1%}")
except ImportError:
    print("Esta célula usa o envio de arquivos do Google Colab.")

## 13. Conclusões

- A **MLP** trata a imagem como uma fila de 3.072 números e perde a noção de vizinhança: precisa de muitos parâmetros e acerta pouco.
- A **CNN** usa **campos receptivos locais**, **pesos compartilhados** e **pooling**: aprende filtros que funcionam em qualquer posição da imagem, com menos parâmetros e muito mais acerto.
- **Dropout** e **early stopping** ajudam a CNN a generalizar em vez de decorar.
- As camadas mais profundas trocam "onde" por "o quê": as características ficam cada vez mais abstratas.

### Exercícios para a turma

1. **Overfitting na prática:** remova a camada `Dropout` e o early stopping, treine por 20 épocas e compare as curvas de treino e validação.
2. **Filtros:** troque 32/64/128 por 16/32/64. O que acontece com o número de parâmetros e com a acurácia?
3. **Sem pooling:** retire o `pool3` e veja o número de parâmetros da camada densa explodir no `summary()`.
4. **Taxa de aprendizado:** troque `optimizer="adam"` por `keras.optimizers.SGD(learning_rate=0.01)` e compare a velocidade de aprendizado.
5. **MLP maior:** dobre os neurônios da MLP. Ela chega perto da CNN?